# Perros vs. gatos: clasificación de imágenes con una RNA

## Objetivo

Entrenar una red neuronal convolucional (`EfficientNet-B0`) que distinga fotos de perros y gatos del dataset *Asirra* (Petfinder.com y Microsoft). Las funciones reutilizables viven en `app.py` y `utils.py`; este notebook las ejecuta paso a paso y muestra los resultados.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from app import (
    BEST_MODEL_PATH,
    CLASSES,
    EPOCHS,
    build_callbacks,
    build_generators,
    build_model,
    evaluate_model,
    prepare_dataset,
    sample_images,
)
from utils import plot_image_grid, plot_training_history

dataset_path = prepare_dataset()
for split in ("train", "test"):
    counts = {label: len(list((dataset_path / split / label).glob("*.jpg"))) for label in CLASSES}
    print(split, counts)

## Paso 1: carga del conjunto de datos

`prepare_dataset()` descarga el zip (~580 MB) y extrae las 25.000 fotos directamente en `data/interim/dogs-vs-cats/{train,test}/{cat,dog}`. La etiqueta está en el nombre del archivo (`cat.12.jpg`, `dog.7.jpg`); al moverla al nombre de la carpeta, `flow_from_directory()` puede etiquetar las imágenes automáticamente. El 20% de cada clase se reserva para test con una semilla fija, de modo que la división es reproducible.

## Paso 2: visualización de la información de entrada

In [ ]:
plot_image_grid(sample_images("dog"), "Primeros 9 perros")
plot_image_grid(sample_images("cat"), "Primeros 9 gatos")

Las fotos son a color y cada una tiene un tamaño distinto (se indica bajo el nombre). Antes de entrar a la red deben tener todas el mismo tamaño. Usamos **224×224 píxeles** en lugar de 200×200 porque es la entrada que declara la arquitectura `EfficientNetB0(input_shape=(224, 224, 3))` del paso 3.

Como el equipo tiene menos de 12 GB de RAM, las imágenes se cargan por lotes con `ImageDataGenerator` y `flow_from_directory()`. Además del train y el test, se separa un 10% del train como **validación**: `EarlyStopping` y `ModelCheckpoint` toman sus decisiones con ella, y el test queda intacto para la medición final.

In [ ]:
trdata, valdata, tsdata = build_generators()
print("Índices de clase:", trdata.class_indices)
print(f"Train: {trdata.samples} | Validación: {valdata.samples} | Test: {tsdata.samples}")

## Paso 3: construcción de la RNA

La arquitectura carga la *backbone* convolucional de `EfficientNet-B0` sin pesos preentrenados (`weights=None`), reduce los mapas de características con `GlobalAveragePooling2D` y termina con dos capas densas. La última tiene 2 neuronas con `softmax`: una probabilidad por clase. El modelo se compila con el optimizador Adam y la pérdida `categorical_crossentropy`.

In [ ]:
model = build_model()
model.summary()

## Paso 4: entrenamiento y optimización

`ModelCheckpoint` guarda en disco el mejor modelo visto durante el entrenamiento y `EarlyStopping` detiene el proceso cuando la validación deja de mejorar. En Keras 3 `fit_generator` ya no existe: `model.fit` acepta los generadores directamente.

> ⚠️ Con las 25.000 imágenes y la red entrenada desde cero, esta celda puede tardar varias horas en un portátil.

In [ ]:
history = model.fit(
    trdata,
    validation_data=valdata,
    epochs=EPOCHS,
    callbacks=build_callbacks(),
)
plot_training_history(history.history)

### Evaluación del mejor modelo con el conjunto de test

In [ ]:
results = evaluate_model(BEST_MODEL_PATH, tsdata)
print(f"Test loss: {results['test_loss']:.4f}")
print(f"Test accuracy: {results['test_accuracy']:.3f}")
display(pd.DataFrame(results["report"]).transpose())
display(pd.DataFrame(
    results["confusion_matrix"],
    index=[f"real_{label}" for label in CLASSES],
    columns=[f"pred_{label}" for label in CLASSES],
))

## Paso 5: modelo guardado

El mejor modelo queda almacenado en `models/efficientnet_dogs_vs_cats.keras` y puede recuperarse con `keras.models.load_model()`.

## Conclusiones

_Completar tras el entrenamiento: precisión en test comparada con el 80% del SVM de 2007, época en la que se detuvo `EarlyStopping`, y si las curvas muestran sobreajuste (train mejora mientras validación empeora)._